# Lab 4 — The Transformer: the building blocks

In Lab 2 we processed sequences with recurrent networks: they read **one token at a time** and squeeze everything
they have seen into a hidden state (→ `rnn_tools.ipynb`). The Transformer ("Attention is all you need", 2017) drops
recurrence altogether: with **attention**, every token can look directly at every other token, all in parallel.

```
           ENCODER (× N)                         DECODER (× N)
  input  ─► embedding + positions          output so far ─► embedding + positions
            │                                               │
            ├─► self-attention                              ├─► masked self-attention
            ├─► feed-forward                                ├─► cross-attention ◄── encoder output ("memory")
            ▼                                               ├─► feed-forward
         memory ─────────────────────────────────────────►  ▼
                                                         linear ─► next-token probabilities
```

We will look at all the pieces on a small but real sequence-to-sequence task, the same one you will train in
`transformer_dates.py`: translating dates written by humans into ISO format.

1. From text to tensors: tokens, padding, embeddings
2. Positional encoding
3. Attention
4. Masks: padding and causal
5. Multi-head attention
6. The other ingredients: feed-forward, residual connections, LayerNorm
7. Encoder block and decoder block
8. Training: teacher forcing
9. Inference: generating one token at a time

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn

from transformer_dates import (EOS, PAD, SOS, SRC_VOCAB, TGT_VOCAB, DateDataset, DecoderBlock, EncoderBlock, collate,
                               decode_tgt, encode_src, encode_tgt)

torch.manual_seed(0)

dataset = DateDataset(6, seed=0)
for src, tgt in dataset.pairs:
    print(f"{src!r:34s} -> {tgt!r}")

---
## 1. From text to tensors

We work at the level of **characters**: each character is a token. We have two vocabularies, one for the input
(letters, digits, punctuation) and one for the output (digits and `-`), both starting with three **special tokens**:
- `<pad>` (0): fills the shorter sequences of a batch;
- `<sos>` (1): "start of sequence", the first input of the decoder;
- `<eos>` (2): "end of sequence", tells us when the decoder has finished writing.

In [ ]:
print("Source vocabulary:", len(SRC_VOCAB), SRC_VOCAB)
print("Target vocabulary:", len(TGT_VOCAB), TGT_VOCAB)

text, iso = dataset.pairs[1]
print(f"\n{text!r} -> {encode_src(text).tolist()}")
print(f"{iso!r} -> {encode_tgt(iso).tolist()}   (with <sos> and <eos>)")

Sequences have different lengths: to put them in a batch we **pad** them to the length of the longest one
(the `collate` function of the script, used by the `DataLoader`, → training §2.5).

In [ ]:
src_batch, tgt_batch = collate([dataset[i] for i in range(4)])
print("src batch:", tuple(src_batch.shape))
print(src_batch)
print("\ntgt batch:", tuple(tgt_batch.shape))
print(tgt_batch)

Finally, an `nn.Embedding` turns every token id into a learned vector of size $D$ (`d_model`): a batch of ids of
shape $(B, L)$ becomes a tensor $(B, L, D)$. From now on, every layer of the Transformer maps $(B, L, D)$ to $(B, L, D)$.

In [ ]:
D = 16
embed = nn.Embedding(len(SRC_VOCAB), D, padding_idx=PAD)
x = embed(src_batch)
print(f"{tuple(src_batch.shape)} -> {tuple(x.shape)}")
print("The <pad> embedding is fixed to zero:", embed.weight[PAD].abs().sum().item())

---
## 2. Positional encoding

Attention compares tokens with each other, but it has **no notion of order**: if we shuffle the input tokens, the
outputs are shuffled in the same way and nothing else changes. For our task this would be a disaster: "march 5" and
"5 march" — or 12.03 and 03.12 — would look the same! Let's verify it with PyTorch's attention layer:

In [ ]:
attn = nn.MultiheadAttention(D, num_heads=2, batch_first=True)
x = torch.randn(1, 5, D)
perm = torch.tensor([3, 0, 4, 1, 2])

out, _ = attn(x, x, x)
out_perm, _ = attn(x[:, perm], x[:, perm], x[:, perm])
print("Shuffled input -> same outputs, just shuffled:", torch.allclose(out[:, perm], out_perm, atol=1e-6))

The fix: we **add** to each embedding a vector that depends on its position. The original paper uses fixed sines and
cosines of different frequencies:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/D}}\right) \qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/D}}\right)$$

The first dimensions oscillate quickly, the last ones slowly: together they give every position a unique
"fingerprint", and nearby positions get similar vectors.

In [ ]:
def positional_encoding(max_len, d_model):
    pos = torch.arange(max_len).unsqueeze(1)                                           # (max_len, 1)
    div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))     # (d_model / 2,)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(pos * div)                                                 # broadcasting (→ tensors §9)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


pe = positional_encoding(50, 64)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
im = ax1.imshow(pe, cmap="RdBu", aspect="auto")
ax1.set_xlabel("dimension")
ax1.set_ylabel("position")
ax1.set_title("Positional encoding (50 positions, D = 64)")
fig.colorbar(im, ax=ax1)
for dim in [0, 4, 16, 40]:
    ax2.plot(pe[:, dim], label=f"dim {dim}")
ax2.set_xlabel("position")
ax2.legend()
ax2.set_title("Different dimensions, different frequencies")
plt.show()

sim = pe @ pe.T                                   # similarity between the encodings of all pairs of positions
print("Similarity of position 10 with positions 8..12:", sim[10, 8:13].round(decimals=1).tolist())
print("Similarity of position 10 with position 40:    ", round(sim[10, 40].item(), 1))

In the model: `x = embedding(tokens) * sqrt(D) + PE[:L]`.

⚠️ **The two terms must have a comparable size.** The positional encoding is in $[-1, 1]$. As in the paper, the
embeddings are initialized with a small standard deviation ($1/\sqrt{D}$) and multiplied by $\sqrt{D}$, so they also
have values of order 1. If instead we kept the default initialization of `nn.Embedding` (std 1) and still multiplied
by $\sqrt{D}$, the embeddings would be ~$\sqrt{D}$ times larger and would **hide the position information**. We saw
it happen while preparing this lab: on the dates task the model got stuck at ~70% of correct dates, writing the right
digits in the wrong order (2021 → 2012). With the right initialization it reaches ~100% in a few epochs.

---
## 3. Attention

Attention is a **soft dictionary lookup**. Each token produces three vectors:
- a **query** $q$: "what am I looking for?"
- a **key** $k$: "what do I contain?"
- a **value** $v$: "what do I pass on if you pick me?"

The query is compared with all the keys with a dot product (a large dot product = similar vectors); a softmax turns
the similarities into weights that sum to 1; the output is the weighted average of the values.

A tiny example with 3 keys, where the query is similar to the second key:

In [ ]:
keys = torch.tensor([[1., 0., 0.],
                     [0., 1., 0.],
                     [0., 0., 1.]])
values = torch.tensor([[10., 0.],
                       [0., 10.],
                       [5., 5.]])
query = torch.tensor([0.2, 3.0, 0.1])

scores = keys @ query                      # one similarity score per key
weights = torch.softmax(scores, dim=0)
output = weights @ values                  # weighted average of the values
print("scores: ", scores.tolist())
print("weights:", weights.round(decimals=3).tolist())
print("output: ", output.round(decimals=2).tolist(), " ~ the value of key 2")

In a Transformer queries, keys and values are computed from the tokens by three learned linear layers,
$Q = X W_Q$, $K = X W_K$, $V = X W_V$, and all the queries are processed at once with matrix products:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

$Q K^T$ has shape $(L_q, L_k)$: row $i$ contains the scores of query $i$ against all the keys.

### Why divide by $\sqrt{d}$?
The dot product of two random vectors of size $d$ grows like $\sqrt{d}$. With large scores the softmax becomes
almost one-hot ("saturated"): one weight ~1 and all the others ~0, with gradients close to zero. Dividing by
$\sqrt{d}$ keeps the scores in a reasonable range whatever the size:

In [ ]:
for d in [4, 64, 512]:
    q, k = torch.randn(1, d), torch.randn(10, d)
    raw = torch.softmax((q @ k.T)[0], dim=0)
    scaled = torch.softmax((q @ k.T)[0] / math.sqrt(d), dim=0)
    print(f"d={d:4d}: largest weight without scaling {raw.max():.2f}, with scaling {scaled.max():.2f}")

### The tools for TODO 1
👉 Writing `scaled_dot_product_attention(q, k, v, mask)` is **TODO 1** of the exercise. In the script `q`, `k`, `v`
are 4-dimensional: $(B, h, L, d)$, with a batch dimension $B$ and a heads dimension $h$ (see §5). You need only a few
PyTorch operations:

| Operation | What it does |
|---|---|
| `a @ b` | matrix product of the **last two** dimensions; all the leading dimensions ($B$, $h$) are treated as a batch |
| `k.transpose(-2, -1)` | swaps the last two dimensions: $(B, h, L_k, d) \to (B, h, d, L_k)$, i.e. $K^T$ for every sequence and head |
| `math.sqrt(d)` | $\sqrt{d}$, with `d = q.shape[-1]` |
| `torch.softmax(x, dim=-1)` | softmax over the last dimension (the keys): each row sums to 1 |
| `scores.masked_fill(mask, value)` | puts `value` where `mask` is True (see §4) |

Let's see the shapes they produce:

In [ ]:
B, h, Lq, Lk, d = 2, 4, 5, 7, 8
q, k, v = torch.randn(B, h, Lq, d), torch.randn(B, h, Lk, d), torch.randn(B, h, Lk, d)

print("k:", tuple(k.shape), " -> k.transpose(-2, -1):", tuple(k.transpose(-2, -1).shape))

products = q @ k.transpose(-2, -1)
print("q @ k^T:", tuple(products.shape), " -> (B, h, Lq, Lk): one number for every (query, key) pair,",
      "in every sequence of the batch and in every head")

w = torch.softmax(products, dim=-1)
print("softmax over dim=-1: every row sums to 1?", torch.allclose(w.sum(-1), torch.ones(B, h, Lq)))

print("(B, h, Lq, Lk) @ v", tuple(v.shape), "->", tuple((w @ v).shape), " (B, h, Lq, d)")

PyTorch also has a ready-made (and very optimized) version of the whole attention, `F.scaled_dot_product_attention`:
the script uses it to **check** your implementation, so your function must give the same output.

In [ ]:
out = F.scaled_dot_product_attention(q, k, v)
print(f"q {tuple(q.shape)}, k {tuple(k.shape)}, v {tuple(v.shape)}  ->  out {tuple(out.shape)}")

---
## 4. Masks

Sometimes a query must **not** look at some keys. We handle it with a boolean mask (in our code **True = ignore**):
the corresponding scores are set to $-\infty$ *before* the softmax, so they get weight exactly 0.

In [ ]:
scores = torch.randn(4, 4)
mask = torch.tensor([False, False, True, True])            # ignore the last two keys
weights = torch.softmax(scores.masked_fill(mask, float("-inf")), dim=-1)
print(weights.round(decimals=2))
print("rows still sum to 1:", weights.sum(-1).tolist())

We need two kinds of masks:

**Padding mask** (encoder and cross-attention): the `<pad>` tokens are not real, nobody must attend to them.
Shape `(B, 1, 1, Ls)`, so that it broadcasts over the heads and over all the queries.

**Causal mask** (decoder self-attention): when the decoder produces character $t$, it must not see characters
$t+1, t+2, \dots$ that it has not generated yet. Query $i$ can only look at keys $j \le i$: the upper triangle is
masked. Combined with the padding of the target: shape `(B, 1, Lt, Lt)`.

In [ ]:
src_mask = (src_batch == PAD)[:, None, None, :]
print("padding mask:", tuple(src_mask.shape))

tgt_in = tgt_batch[:, :-1]
Lt = tgt_in.shape[1]
causal = torch.triu(torch.ones(Lt, Lt, dtype=torch.bool), diagonal=1)
tgt_mask = causal[None, None] | (tgt_in == PAD)[:, None, None, :]
print("target mask: ", tuple(tgt_mask.shape))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(src_mask[:, 0, 0], cmap="Greys")
axes[0].set_title("Padding mask of the source batch (black = ignore)")
axes[0].set_xlabel("key position")
axes[0].set_ylabel("sequence in the batch")
axes[1].imshow(causal, cmap="Greys")
axes[1].set_title("Causal mask (black = ignore)")
axes[1].set_xlabel("key position (what I look at)")
axes[1].set_ylabel("query position (who is looking)")
plt.show()

### Applying a mask to a whole batch of scores
In the attention function the scores have shape $(B, h, L_q, L_k)$, while the padding mask is $(B, 1, 1, L_k)$: thanks
to **broadcasting** (→ tensors §9) `masked_fill` applies the same mask to every head and every query. This is the
masking step you need in TODO 1:

In [ ]:
h = 4
scores = torch.randn(4, h, 6, src_batch.shape[1])            # (B, h, Lq, Ls): e.g. 6 queries against the source
print("scores:", tuple(scores.shape), "  mask:", tuple(src_mask.shape))

masked = scores.masked_fill(src_mask, float("-inf"))         # (B, 1, 1, Ls) broadcasts to (B, h, Lq, Ls)
w = torch.softmax(masked, dim=-1)

n_real = int((src_batch[2] != PAD).sum())                    # the 3rd sequence has n_real real characters
print(f"3rd sequence: {n_real} real characters, then padding")
print("weights of one query on the real keys:  ", w[2, 0, 0, :n_real].sum().item())
print("weights of the same query on the padding:", w[2, 0, 0, n_real:].sum().item())

⚠️ **Careful with conventions**: in our code (and in `nn.MultiheadAttention`'s `key_padding_mask`) `True` means
*ignore*, while in `F.scaled_dot_product_attention` a boolean `attn_mask` means *attend*. Always check the
documentation!

In [ ]:
q, k, v = torch.randn(1, 1, 3, 8), torch.randn(1, 1, 4, 8), torch.randn(1, 1, 4, 8)
our_mask = torch.tensor([False, False, False, True]).view(1, 1, 1, 4)   # ignore the last key
out_pytorch = F.scaled_dot_product_attention(q, k, v, attn_mask=~our_mask)   # invert it for PyTorch
out_nomask = F.scaled_dot_product_attention(q, k, v)
print("The mask changes the result:", not torch.allclose(out_pytorch, out_nomask))

---
## 5. Multi-head attention

A single attention computes one weighted average per token: it can focus on one "kind" of relation at a time.
**Multi-head attention** runs $h$ attentions in parallel, each on a slice of size $d = D / h$ of the projected
vectors, then concatenates the results and mixes them with a last linear layer $W_O$. Different heads can learn
different relations (e.g. one looks at the month, another at the day).

In practice the heads are just a reshape: $(B, L, D) \rightarrow (B, L, h, d) \rightarrow (B, h, L, d)$
(→ tensors §6), so that the attention runs on all the heads at once.

In [ ]:
B, L, D, h = 2, 6, 16, 4
x = torch.randn(B, L, D)
W_q = nn.Linear(D, D)

q = W_q(x).view(B, L, h, D // h).transpose(1, 2)        # split into heads
print(f"{tuple(x.shape)} -> {tuple(q.shape)}   (B, h, L, d)")

merged = q.transpose(1, 2).contiguous().view(B, L, D)   # and back: concatenate the heads
print(f"{tuple(q.shape)} -> {tuple(merged.shape)}")
print("Same numbers:", torch.allclose(merged, W_q(x)))

The class `MultiHeadAttention` in `transformer_dates.py` is ready: it does exactly this, calling your
`scaled_dot_product_attention` in the middle. You will call it inside the blocks as

```python
out = attn(query, key, value, mask)      # query (B, Lq, D), key and value (B, Lk, D), mask True = ignore  ->  (B, Lq, D)
```

It is used in two ways:
- **self-attention**: `attn(x, x, x, mask)` — queries, keys and values come from the same sequence;
- **cross-attention**: `attn(y, memory, memory, mask)` — the queries come from the decoder, keys and values from the
  output of the encoder. The result has the length of the **queries**.

PyTorch's `nn.MultiheadAttention` works the same way (but it also returns the attention weights, and takes the masks
as separate arguments):

In [ ]:
mha = nn.MultiheadAttention(D, h, batch_first=True)      # PyTorch's version, same interface idea
y, memory = torch.randn(B, 3, D), torch.randn(B, 7, D)
out, weights = mha(y, memory, memory)
print(f"queries {tuple(y.shape)}, keys/values {tuple(memory.shape)} -> out {tuple(out.shape)}, "
      f"weights {tuple(weights.shape)} (averaged over the heads)")

---
## 6. The other ingredients of a block

**Feed-forward network**: after attention has mixed information *between* tokens, a small MLP processes *each token
separately* (the same weights for every position): `Linear(D, d_ff) → ReLU → Linear(d_ff, D)`. `nn.Linear` acts on the
last dimension, so it works directly on $(B, L, D)$.

**Residual connection**: each sub-layer computes a *correction* that is added to its input, $x + \text{sublayer}(x)$.
The gradient can flow directly through the sum, which makes deep stacks trainable (the same idea as in ResNet).

**LayerNorm**: normalizes **each token** over its $D$ features (mean 0, std 1, then a learned scale and shift).
Unlike BatchNorm (→ `cnn_tools.ipynb` §3) it does not depend on the other elements of the batch, nor on the
sequence length.

In [ ]:
x = torch.randn(2, 5, D) * 3 + 7
ln = nn.LayerNorm(D)
out = ln(x)
print("per-token mean before:", x.mean(-1)[0].round(decimals=2).tolist())
print("per-token mean after: ", out.mean(-1)[0].round(decimals=2).tolist())
print("per-token std after:  ", out.std(-1, unbiased=False)[0].round(decimals=2).tolist())

Putting them together, each sub-layer of the Transformer follows the same pattern (as in the original paper):

$$x \leftarrow \text{LayerNorm}\big(x + \text{Dropout}(\text{sublayer}(x))\big)$$

Here it is with a generic sub-layer (a simple `nn.Linear`): the shape never changes, so sub-layers can be chained.

```python
x = norm(x + dropout(sublayer(x)))
```

In [ ]:
sublayer, norm, dropout = nn.Linear(D, D), nn.LayerNorm(D), nn.Dropout(0.1)
x = torch.randn(2, 7, D)
x_new = norm(x + dropout(sublayer(x)))
print(f"{tuple(x.shape)} -> {tuple(x_new.shape)}")

---
## 7. Encoder block and decoder block

**Encoder block** — two sub-layers:
1. self-attention over the source (with the padding mask);
2. feed-forward.

**Decoder block** — three sub-layers:
1. **masked** self-attention over the target produced so far (with the causal mask);
2. **cross-attention**: queries from the decoder, keys and values from the encoder output ("memory"), with the
   padding mask of the source. This is where the decoder reads the input;
3. feed-forward.

Each sub-layer is wrapped in the residual + LayerNorm pattern above, and the shape never changes. The full model
stacks $N$ encoder blocks and $N$ decoder blocks, and a final linear layer turns each decoder output into scores
over the target vocabulary.

### The tools for TODO 2 and TODO 3
In the script the layers of the two blocks are **already created** in `__init__`: TODO 2 and TODO 3 are only their
`forward`, where you combine them. These are the pieces at your disposal:

| Attribute | Block | How to call it | Output |
|---|---|---|---|
| `self.self_attn` | encoder, decoder | `self.self_attn(query, key, value, mask)` | $(B, L_q, D)$ |
| `self.cross_attn` | decoder | `self.cross_attn(query, key, value, mask)` | $(B, L_q, D)$ |
| `self.ffn` | encoder, decoder | `self.ffn(x)` | same as the input |
| `self.norm1`, `self.norm2` (`self.norm3`) | encoder (decoder) | `self.norm1(x)` — one per sub-layer | same as the input |
| `self.dropout` | encoder, decoder | `self.dropout(x)` — on the output of each sub-layer | same as the input |

And the inputs of the two `forward`:

| Argument | Shape | What it is |
|---|---|---|
| `x` (encoder) / `y` (decoder) | $(B, L_s, D)$ / $(B, L_t, D)$ | the embedded source / target tokens |
| `memory` (decoder) | $(B, L_s, D)$ | the output of the encoder |
| `src_mask` | $(B, 1, 1, L_s)$ | True on the padding of the **source** |
| `tgt_mask` (decoder) | $(B, 1, L_t, L_t)$ | True on the **future** positions and on the padding of the target |

Think about it: for each attention, what are the queries, what are the keys and values, and which mask applies?

In [ ]:
print(EncoderBlock(d_model=D, n_heads=4, d_ff=32))
print(DecoderBlock(d_model=D, n_heads=4, d_ff=32))

For reference, PyTorch has ready-made versions of the two blocks (`nn.TransformerEncoderLayer`,
`nn.TransformerDecoderLayer`) and of the whole model (`nn.Transformer`); in the exercise you build your own.

In [ ]:
enc_layer = nn.TransformerEncoderLayer(d_model=D, nhead=4, dim_feedforward=32, batch_first=True)
dec_layer = nn.TransformerDecoderLayer(d_model=D, nhead=4, dim_feedforward=32, batch_first=True)
src, tgt = torch.randn(2, 7, D), torch.randn(2, 3, D)
memory = enc_layer(src)
print("encoder layer:", tuple(src.shape), "->", tuple(memory.shape))
print("decoder layer:", tuple(tgt.shape), "+ memory", tuple(memory.shape), "->", tuple(dec_layer(tgt, memory).shape))

---
## 8. Training: teacher forcing

During training we know the whole target, so we do not need to generate it step by step. We feed the decoder the
target **shifted right** (starting with `<sos>`) and ask it to predict, at each position, the **next** character:

In [ ]:
tgt = encode_tgt("2021-03-05")
tgt_in, tgt_out = tgt[:-1], tgt[1:]
print(f"{'position':>8s} | {'decoder input':>13s} | {'must predict':>12s}")
for t, (i, o) in enumerate(zip(tgt_in.tolist(), tgt_out.tolist())):
    print(f"{t:8d} | {TGT_VOCAB[i]:>13s} | {TGT_VOCAB[o]:>12s}")

Thanks to the **causal mask**, position $t$ only sees the inputs up to $t$: all the positions are trained **in
parallel** in a single forward pass, without cheating. (This is called *teacher forcing*, because the decoder always
receives the correct previous characters.)

The loss is a cross-entropy over the target vocabulary at every position, ignoring the padding
(`nn.CrossEntropyLoss(ignore_index=PAD)`):

```python
logits = model(src, tgt[:, :-1])                                     # (B, Lt, |V|)
loss = loss_fn(logits.reshape(-1, len(TGT_VOCAB)), tgt[:, 1:].reshape(-1))
```

---
## 9. Inference: generating one token at a time

At test time the target is unknown. The decoder generates it **autoregressively**: start from `<sos>`, predict the
next character, append it to the input, and repeat until `<eos>`. The encoder runs only once. Taking the most likely
character at each step is called **greedy decoding**. Here with an untrained `nn.Transformer`, just to see the loop:

In [ ]:
model = nn.Transformer(d_model=D, nhead=h, num_encoder_layers=1, num_decoder_layers=1, dim_feedforward=32,
                       batch_first=True)
src_embed, tgt_embed, head = nn.Embedding(len(SRC_VOCAB), D), nn.Embedding(len(TGT_VOCAB), D), nn.Linear(D, len(TGT_VOCAB))
model.eval()

src = encode_src("march 5, 2021").unsqueeze(0)
with torch.no_grad():
    memory = model.encoder(src_embed(src))                     # encode once
    ys = torch.tensor([[SOS]])
    for step in range(11):
        causal = nn.Transformer.generate_square_subsequent_mask(ys.shape[1])
        out = model.decoder(tgt_embed(ys), memory, tgt_mask=causal)
        next_token = head(out[:, -1]).argmax(-1, keepdim=True)  # most likely next character
        ys = torch.cat([ys, next_token], dim=1)
        print(f"step {step:2d}: {[TGT_VOCAB[i] for i in ys[0].tolist()]}")
        if next_token.item() == EOS:
            break

print("\nDecoded (random, the model is untrained!):", repr(decode_tgt(ys[0].tolist())))

---
## Summary

| Piece | What it does | Shape |
|---|---|---|
| `nn.Embedding` | token id → vector | $(B, L) \to (B, L, D)$ |
| positional encoding | adds order information | $(B, L, D)$ |
| scaled dot-product attention | $\text{softmax}(QK^T/\sqrt{d})V$ | $(B, h, L_q, d)$ |
| padding mask | ignore `<pad>` | $(B, 1, 1, L_s)$ |
| causal mask | do not look at the future | $(B, 1, L_t, L_t)$ |
| multi-head attention | $h$ attentions in parallel + $W_O$ | $(B, L_q, D)$ |
| feed-forward | MLP on each token | $(B, L, D)$ |
| residual + LayerNorm | $\text{LN}(x + \text{sublayer}(x))$ | $(B, L, D)$ |
| encoder block | self-attn + FFN | $(B, L_s, D)$ |
| decoder block | masked self-attn + cross-attn + FFN | $(B, L_t, D)$ |

👉 **Now it's your turn:** open `transformer_dates.py` and complete the three TODOs. The script checks each of them
before training, and at the end it shows the **cross-attention map**: which input characters the decoder looks at
while writing each output character.